# LDA: Temporal Topic Analysis

Analyze topic evolution over time (2000–2025) using **existing best LDA models**
trained on all documents. Same approach as Top2Vec temporal analysis.

1. Get dominant topic per document from the single trained model
2. Group documents by year
3. Compute per-year topic prevalence, c-TF-IDF word evolution, coherence & IRBO

**Topics are consistent across all years** — no alignment needed.

In [1]:
import gc
import time
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Configuration

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]

BASE_DIR = Path("../../../../data/preprocess")
MODEL_DIR = Path("../../../../models/lda/tuning")
RESULT_DIR = Path("../../../../results/lda/temporal")
VERSION = "v1"

# IRBO config
TOP_N_WORDS = 10
RBO_P = 0.9

# Create output directories
for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Subjects: {LIST_SUBJECT}")
print(f"Model directory: {MODEL_DIR}")
print(f"Results directory: {RESULT_DIR}")

Subjects: ['cs', 'math', 'physics']
Model directory: ../../../../models/lda/tuning
Results directory: ../../../../results/lda/temporal


## Helper Functions

In [3]:
def load_dataset(subject: str) -> pd.DataFrame:
    """Load dataset with year column."""
    file_path = BASE_DIR / subject / "emb" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    df["submitted_date"] = pd.to_datetime(df["submitted_date"])
    df["year"] = df["submitted_date"].dt.year
    return df


def get_dominant_topics(model, corpus):
    """
    Get the dominant (highest probability) topic for each document.
    Returns array of topic IDs.
    """
    dominant_topics = []
    for doc_bow in corpus:
        topic_dist = model.get_document_topics(doc_bow, minimum_probability=0.0)
        if topic_dist:
            dominant = max(topic_dist, key=lambda x: x[1])[0]
        else:
            dominant = 0
        dominant_topics.append(dominant)
    return np.array(dominant_topics)


def compute_ctfidf_per_year(df, topic_col="topic", text_col="text", top_n=10):
    """
    Compute c-TF-IDF per topic per year.
    Groups docs by (year, topic), concatenates text, applies TF-IDF.
    """
    groups = df.groupby(["year", topic_col])[text_col].apply(
        lambda x: " ".join(x)
    ).reset_index()
    groups.columns = ["year", "topic", "text"]

    vectorizer = CountVectorizer(stop_words="english")
    tf_matrix = vectorizer.fit_transform(groups["text"])
    vocab = vectorizer.get_feature_names_out()

    n_groups = tf_matrix.shape[0]
    df_t = (tf_matrix > 0).sum(axis=0).A1
    idf = np.log((n_groups + 1) / (df_t + 1)) + 1

    tfidf_matrix = tf_matrix.multiply(idf).toarray()
    row_norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
    row_norms[row_norms == 0] = 1
    tfidf_matrix = tfidf_matrix / row_norms

    topic_words_per_year = {}
    for idx, row in groups.iterrows():
        year = row["year"]
        topic = row["topic"]
        scores = tfidf_matrix[idx]
        top_indices = scores.argsort()[-top_n:][::-1]
        top_words = [vocab[i] for i in top_indices if scores[i] > 0]
        topic_words_per_year[(year, topic)] = top_words

    return topic_words_per_year


def calculate_coherence_for_words(topic_word_lists, texts_tokenized, dictionary):
    """Calculate C_v coherence given a list of topic word lists."""
    valid_topics = [tw for tw in topic_word_lists if len(tw) >= 2]
    if len(valid_topics) == 0:
        return 0.0
    cm = CoherenceModel(
        topics=valid_topics,
        texts=texts_tokenized,
        dictionary=dictionary,
        coherence='c_v',
        processes=1
    )
    return cm.get_coherence()


def rbo(list_1, list_2, p=0.9):
    """Rank-Biased Overlap."""
    k = min(len(list_1), len(list_2))
    if k == 0:
        return 0.0
    rbo_score = 0.0
    for d in range(1, k + 1):
        set_1 = set(list_1[:d])
        set_2 = set(list_2[:d])
        agreement = len(set_1 & set_2) / d
        rbo_score += (p ** (d - 1)) * agreement
    rbo_score *= (1 - p)
    return rbo_score


def calculate_irbo(topics_words, p=0.9):
    """Calculate mean IRBO diversity."""
    if len(topics_words) < 2:
        return 0.0
    irbo_scores = []
    for (i, j) in combinations(range(len(topics_words)), 2):
        similarity = rbo(topics_words[i], topics_words[j], p=p)
        irbo_scores.append(1.0 - similarity)
    return np.mean(irbo_scores)

## Load Models & Data

Load each best LDA model, reconstruct the BoW corpus using the model's dictionary,
then get the dominant topic for every document.

In [4]:
all_models = {}
all_data = {}
all_years = {}

for subject in LIST_SUBJECT:
    print(f"\nLoading {subject}...")

    # Load model
    model_path = MODEL_DIR / subject / "best_model.pkl"
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    all_models[subject] = model

    # Load data
    df = load_dataset(subject)

    # Reconstruct BoW corpus using model's id2word dictionary
    print(f"  Building BoW corpus with model's dictionary ({len(model.id2word)} words)...")
    start = time.time()
    texts_tokenized = [text.split() for text in df["text"].tolist()]
    corpus = [model.id2word.doc2bow(tokens) for tokens in texts_tokenized]

    # Get dominant topic per document
    print(f"  Getting dominant topics for {len(df):,} documents...")
    dominant_topics = get_dominant_topics(model, corpus)
    df["topic"] = dominant_topics
    elapsed = time.time() - start

    all_data[subject] = df
    years = sorted(df["year"].unique())
    all_years[subject] = years

    n_topics = model.num_topics
    print(f"  {subject}: {len(df):,} docs, {n_topics} topics, "
          f"{len(years)} years ({years[0]}-{years[-1]}) [{elapsed:.1f}s]")

    # Topic size distribution
    topic_sizes = df["topic"].value_counts().sort_index()
    print(f"  Topic sizes: min={topic_sizes.min()}, "
          f"max={topic_sizes.max()}, mean={topic_sizes.mean():.0f}")

    del corpus, texts_tokenized
    gc.collect()

print(f"\n✅ All subjects loaded")


Loading cs...
  Building BoW corpus with model's dictionary (21694 words)...
  Getting dominant topics for 165,756 documents...
  cs: 165,756 docs, 75 topics, 26 years (2000-2025) [102.8s]
  Topic sizes: min=9, max=10659, mean=2240

Loading math...
  Building BoW corpus with model's dictionary (16835 words)...
  Getting dominant topics for 157,085 documents...
  math: 157,085 docs, 50 topics, 26 years (2000-2025) [65.1s]
  Topic sizes: min=55, max=9145, mean=3142

Loading physics...
  Building BoW corpus with model's dictionary (21894 words)...
  Getting dominant topics for 146,311 documents...
  physics: 146,311 docs, 50 topics, 26 years (2000-2025) [67.0s]
  Topic sizes: min=76, max=7629, mean=2926

✅ All subjects loaded


## Topic Prevalence Over Time

For each year, compute the proportion of documents belonging to each topic.

In [5]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    n_topics = model.num_topics

    prevalence_csv = RESULT_DIR / subject / "topic_prevalence.csv"

    print(f"\n{'='*70}")
    print(f"Topic Prevalence: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    # Get global topic words from LDA model
    global_topic_words = {}
    for tid in range(n_topics):
        words = [w for w, _ in model.show_topic(tid, topn=5)]
        global_topic_words[tid] = ", ".join(words)

    prevalence_rows = []

    for year in years:
        year_df = df[df["year"] == year]
        n_docs_year = len(year_df)
        topic_counts = year_df["topic"].value_counts()

        for topic_id in range(n_topics):
            count = topic_counts.get(topic_id, 0)
            proportion = count / n_docs_year if n_docs_year > 0 else 0.0

            prevalence_rows.append({
                "subject": subject,
                "year": year,
                "topic_id": topic_id,
                "doc_count": count,
                "total_docs_year": n_docs_year,
                "proportion": round(proportion, 6),
                "top_words": global_topic_words[topic_id],
            })

        active_topics = (topic_counts > 0).sum()
        top_topic = topic_counts.idxmax()
        top_count = topic_counts.max()
        print(f"  {year}: {n_docs_year:,} docs, {active_topics}/{n_topics} active, "
              f"top=T{top_topic} ({top_count} docs)")

    prevalence_df = pd.DataFrame(prevalence_rows)
    prevalence_df.to_csv(prevalence_csv, index=False)
    print(f"\n  Saved: {prevalence_csv} ({len(prevalence_df)} rows)")


Topic Prevalence: CS (75 topics)
  2000: 488 docs, 45/75 active, top=T7 (109 docs)
  2001: 594 docs, 51/75 active, top=T7 (62 docs)
  2002: 648 docs, 56/75 active, top=T7 (97 docs)
  2003: 825 docs, 54/75 active, top=T33 (105 docs)
  2004: 948 docs, 62/75 active, top=T24 (111 docs)
  2005: 1,000 docs, 54/75 active, top=T63 (101 docs)
  2006: 1,000 docs, 63/75 active, top=T63 (111 docs)
  2007: 1,000 docs, 58/75 active, top=T66 (115 docs)
  2008: 1,000 docs, 62/75 active, top=T66 (126 docs)
  2009: 1,000 docs, 58/75 active, top=T63 (123 docs)
  2010: 1,362 docs, 63/75 active, top=T66 (142 docs)
  2011: 1,622 docs, 68/75 active, top=T66 (159 docs)
  2012: 2,254 docs, 66/75 active, top=T24 (201 docs)
  2013: 2,719 docs, 68/75 active, top=T24 (233 docs)
  2014: 2,989 docs, 70/75 active, top=T66 (280 docs)
  2015: 3,345 docs, 67/75 active, top=T66 (312 docs)
  2016: 4,280 docs, 68/75 active, top=T66 (331 docs)
  2017: 5,534 docs, 71/75 active, top=T66 (359 docs)
  2018: 7,470 docs, 72/75 a

## Topic Word Evolution (c-TF-IDF per Year)

Compute c-TF-IDF for each (year, topic) to track how topic word
compositions change over time.

In [6]:
all_topic_words_per_year = {}

for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    n_topics = model.num_topics

    evolution_csv = RESULT_DIR / subject / "topic_word_evolution.csv"

    print(f"\n{'='*70}")
    print(f"Topic Word Evolution: {subject.upper()}")
    print(f"{'='*70}")

    start = time.time()
    topic_words_per_year = compute_ctfidf_per_year(
        df, topic_col="topic", text_col="text", top_n=TOP_N_WORDS
    )
    elapsed = time.time() - start
    all_topic_words_per_year[subject] = topic_words_per_year

    print(f"  c-TF-IDF computed in {elapsed:.1f}s")
    print(f"  (year, topic) groups: {len(topic_words_per_year)}")

    evolution_rows = []
    for (year, topic_id), words in sorted(topic_words_per_year.items()):
        evolution_rows.append({
            "subject": subject,
            "year": year,
            "topic_id": topic_id,
            "top_words": ", ".join(words),
        })

    evolution_df = pd.DataFrame(evolution_rows)
    evolution_df.to_csv(evolution_csv, index=False)
    print(f"  Saved: {evolution_csv}")

    # Example: topic 0 across years
    print(f"\n  Example — Topic 0 word evolution:")
    for year in [2000, 2005, 2010, 2015, 2020, 2025]:
        key = (year, 0)
        if key in topic_words_per_year:
            words = ", ".join(topic_words_per_year[key][:5])
            print(f"    {year}: {words}")


Topic Word Evolution: CS
  c-TF-IDF computed in 24.0s
  (year, topic) groups: 1676
  Saved: ../../../../results/lda/temporal/cs/topic_word_evolution.csv

  Example — Topic 0 word evolution:
    2005: image, superresolution, content, information, pnn
    2010: image, images, phong, blurred, based
    2015: image, images, diffusion, resolution, hkmdhe
    2020: image, images, resolution, style, network
    2025: image, diffusion, images, editing, models

Topic Word Evolution: MATH
  c-TF-IDF computed in 9.2s
  (year, topic) groups: 1286
  Saved: ../../../../results/lda/temporal/math/topic_word_evolution.csv

  Example — Topic 0 word evolution:
    2000: functions, identities, function, tricomplex, series
    2005: functions, polynomials, zeta, series, function
    2010: polynomials, functions, function, zeta, polynomial
    2015: functions, polynomials, function, series, modular
    2020: functions, polynomials, function, zeta, series
    2025: functions, polynomials, function, mathbb, 

## Per-Year Coherence & IRBO

For each year, compute coherence and IRBO using the year-specific
c-TF-IDF topic words.

In [7]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    n_topics = model.num_topics
    topic_words_per_year = all_topic_words_per_year[subject]

    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"

    print(f"\n{'='*70}")
    print(f"Per-Year Metrics: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    metrics_rows = []

    for year in years:
        year_df = df[df["year"] == year]
        n_docs = len(year_df)
        texts_tokenized = [text.split() for text in year_df["text"].tolist()]
        dictionary = Dictionary(texts_tokenized)

        active_topics = sorted(year_df["topic"].unique())
        year_topic_words = []
        for tid in active_topics:
            key = (year, tid)
            if key in topic_words_per_year and len(topic_words_per_year[key]) >= 2:
                year_topic_words.append(topic_words_per_year[key])

        coherence = calculate_coherence_for_words(
            year_topic_words, texts_tokenized, dictionary
        )
        irbo_mean = calculate_irbo(year_topic_words, p=RBO_P)

        if coherence + irbo_mean > 0:
            topic_quality = 2 * coherence * irbo_mean / (coherence + irbo_mean)
        else:
            topic_quality = 0.0

        n_active = len(active_topics)
        print(f"  {year}: {n_docs:,} docs, {n_active} active | "
              f"Quality={topic_quality:.4f} (C={coherence:.4f}, IRBO={irbo_mean:.4f})")

        metrics_rows.append({
            "subject": subject,
            "year": year,
            "num_docs": n_docs,
            "num_topics_total": n_topics,
            "num_topics_active": n_active,
            "coherence_cv": round(coherence, 6),
            "irbo_mean": round(irbo_mean, 6),
            "topic_quality": round(topic_quality, 6),
        })

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(metrics_csv, index=False)
    print(f"\n  Saved: {metrics_csv}")


Per-Year Metrics: CS (75 topics)
  2000: 488 docs, 45 active | Quality=0.7012 (C=0.5407, IRBO=0.9972)
  2001: 594 docs, 51 active | Quality=0.7044 (C=0.5446, IRBO=0.9968)
  2002: 648 docs, 56 active | Quality=0.7224 (C=0.5662, IRBO=0.9975)
  2003: 825 docs, 54 active | Quality=0.6657 (C=0.4996, IRBO=0.9972)
  2004: 948 docs, 62 active | Quality=0.6571 (C=0.4898, IRBO=0.9980)
  2005: 1,000 docs, 54 active | Quality=0.6235 (C=0.4535, IRBO=0.9975)
  2006: 1,000 docs, 63 active | Quality=0.6720 (C=0.5063, IRBO=0.9988)
  2007: 1,000 docs, 58 active | Quality=0.6448 (C=0.4760, IRBO=0.9988)
  2008: 1,000 docs, 62 active | Quality=0.5917 (C=0.4206, IRBO=0.9972)
  2009: 1,000 docs, 58 active | Quality=0.6175 (C=0.4468, IRBO=0.9991)
  2010: 1,362 docs, 63 active | Quality=0.6330 (C=0.4636, IRBO=0.9976)
  2011: 1,622 docs, 68 active | Quality=0.6406 (C=0.4717, IRBO=0.9975)
  2012: 2,254 docs, 66 active | Quality=0.5923 (C=0.4215, IRBO=0.9961)
  2013: 2,719 docs, 68 active | Quality=0.6101 (C=0.4

## Topic Trends: Growing, Stable, and Declining

In [8]:
from scipy.stats import linregress

for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    # Get topic words from evolution CSV
    evo_path = RESULT_DIR / subject / "topic_word_evolution.csv"
    evo_df = pd.read_csv(evo_path)
    # Use most recent year's words as representative
    last_year = evo_df['year'].max()
    last_evo = evo_df[evo_df['year'] == last_year]
    global_tw = {}
    for _, erow in last_evo.iterrows():
        global_tw[int(erow['topic_id'])] = [w.strip() for w in str(erow['top_words']).split(',')][:5]

    rows = []
    for tid in range(model.num_topics):
        topic_df = df[df["topic"] == tid]
        if len(topic_df) == 0:
            continue

        year_counts = topic_df["year"].value_counts().sort_index()
        total_per_year = df["year"].value_counts().sort_index()
        proportions = (year_counts / total_per_year).fillna(0)

        # Align proportions with all years
        all_years = sorted(total_per_year.index)
        prop_aligned = proportions.reindex(all_years, fill_value=0.0)

        years_arr = np.array(all_years, dtype=float)
        props_arr = prop_aligned.values.astype(float)

        # Linear regression
        slope, intercept, r_val, p_val, std_err = linregress(years_arr, props_arr)

        # Early/late for display
        topic_years = sorted(year_counts.index)
        early_mean = proportions[topic_years[:5]].mean() if len(topic_years) >= 5 else proportions.mean()
        late_mean = proportions[topic_years[-5:]].mean() if len(topic_years) >= 5 else proportions.mean()

        # Classify by slope significance
        if p_val < 0.05 and slope > 0:
            trend_label = "GROWING"
        elif p_val < 0.05 and slope < 0:
            trend_label = "DECLINING"
        else:
            trend_label = "STABLE"

        top_words = global_tw.get(tid, ["?"])
        rows.append({
            "subject": subject, "topic_id": tid,
            "top_words": ", ".join(top_words),
            "total_docs": len(topic_df),
            "first_year": year_counts.index.min(),
            "last_year": year_counts.index.max(),
            "early_proportion": round(early_mean, 6),
            "late_proportion": round(late_mean, 6),
            "slope": round(slope, 8),
            "r_squared": round(r_val**2, 4),
            "p_value": round(p_val, 6),
            "trend": trend_label,
        })

    trends_df = pd.DataFrame(rows)
    trends_df.to_csv(RESULT_DIR / subject / "topic_trends.csv", index=False)

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])
    print(f"  {subject.upper()}: Growing={g}, Stable={s}, Declining={d}")


  CS: Growing=30, Stable=23, Declining=21
  MATH: Growing=19, Stable=12, Declining=19
  PHYSICS: Growing=17, Stable=15, Declining=18


## Top 5 Growing & Declining Topics

In [9]:
for subject in LIST_SUBJECT:
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")

    print(f"\n{'='*80}")
    print(f"  {subject.upper()}")
    print(f"{'='*80}")

    growing = trends_df[trends_df['trend'] == 'GROWING'].sort_values('slope', ascending=False)
    declining = trends_df[trends_df['trend'] == 'DECLINING'].sort_values('slope', ascending=True)

    print(f"\n  " + chr(0x1F4C8) + f" TOP 5 GROWING (steepest positive slope):")
    for _, row in growing.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} "
              f"R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | "
              f"{row['top_words']}")

    print(f"\n  " + chr(0x1F4C9) + f" TOP 5 DECLINING (steepest negative slope):")
    for _, row in declining.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} "
              f"R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | "
              f"{row['top_words']}")



  CS

  📈 TOP 5 GROWING (steepest positive slope):
    T 49 | slope=+0.002711 R²=0.715 | 0.0018 → 0.0599 | learning, domain, data, supervised, training
    T 58 | slope=+0.002566 R²=0.322 | 0.0008 → 0.0625 | llms, reasoning, models, language, llm
    T 64 | slope=+0.002349 R²=0.531 | 0.0021 → 0.0559 | models, model, training, performance, based
    T 34 | slope=+0.001965 R²=0.849 | 0.0022 → 0.0397 | 3d, scene, view, splatting, object
    T 65 | slope=+0.001917 R²=0.728 | 0.0024 → 0.0370 | features, feature, fusion, attention, detection

  📉 TOP 5 DECLINING (steepest negative slope):
    T  7 | slope=-0.005444 R²=0.727 | 0.1344 → 0.0120 | semantics, logic, language, reasoning, proof
    T 63 | slope=-0.004414 R²=0.776 | 0.0945 → 0.0160 | mathbb, automata, functions, finite, algebraic
    T 24 | slope=-0.002838 R²=0.808 | 0.1038 → 0.0376 | llms, models, llm, data, model
    T 61 | slope=-0.002806 R²=0.700 | 0.0819 → 0.0246 | problem, algorithm, problems, time, optimization
    T 33 | sl

## Evolution Summary

In [10]:
for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    if not metrics_csv.exists():
        continue

    metrics_df = pd.read_csv(metrics_csv)
    trends_df = pd.read_csv(trends_csv)
    n_topics = all_models[subject].num_topics

    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])

    summary_data = {
        "subject": subject,
        "num_topics": n_topics,
        "num_years": len(metrics_df),
        "coherence_mean": round(metrics_df["coherence_cv"].mean(), 6),
        "coherence_std": round(metrics_df["coherence_cv"].std(), 6),
        "irbo_mean": round(metrics_df["irbo_mean"].mean(), 6),
        "irbo_std": round(metrics_df["irbo_mean"].std(), 6),
        "quality_mean": round(metrics_df["topic_quality"].mean(), 6),
        "quality_std": round(metrics_df["topic_quality"].std(), 6),
        "topics_growing": growing,
        "topics_stable": stable,
        "topics_declining": declining,
    }
    summary_df = pd.DataFrame([summary_data])
    summary_csv = RESULT_DIR / subject / "evolution_summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"  {subject.upper()} summary saved: {summary_csv}")

  CS summary saved: ../../../../results/lda/temporal/cs/evolution_summary.csv
  MATH summary saved: ../../../../results/lda/temporal/math/evolution_summary.csv
  PHYSICS summary saved: ../../../../results/lda/temporal/physics/evolution_summary.csv


## Final Results

In [11]:
print("\n" + "=" * 110)
print("LDA TEMPORAL ANALYSIS: FINAL RESULTS")
print("=" * 110)

for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    if not metrics_csv.exists():
        print(f"\n{subject.upper()}: No results found")
        continue

    metrics_df = pd.read_csv(metrics_csv)
    trends_df = pd.read_csv(trends_csv)
    n_topics = all_models[subject].num_topics

    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])

    sep = chr(9472)
    print(f"\n{sep*60}")
    print(f"  Subject:        {subject.upper()}")
    print(f"  Num topics:     {n_topics}")
    print(f"  Years:          {len(metrics_df)}")
    print(f"  Coherence:      {metrics_df['coherence_cv'].mean():.4f} +/- {metrics_df['coherence_cv'].std():.4f}")
    print(f"  IRBO:           {metrics_df['irbo_mean'].mean():.4f} +/- {metrics_df['irbo_mean'].std():.4f}")
    print(f"  Topic Quality:  {metrics_df['topic_quality'].mean():.4f} +/- {metrics_df['topic_quality'].std():.4f}")
    print(f"  Trends:         ↑{growing} growing, →{stable} stable, ↓{declining} declining")
    print(f"{sep*60}")

print("\n" + "=" * 110)
print("Per-Year Details:")
print("=" * 110)

for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    if not metrics_csv.exists():
        continue
    metrics_df = pd.read_csv(metrics_csv)
    print(f"\n{subject.upper()}:")
    print(metrics_df[["year", "num_docs", "num_topics_active",
                     "coherence_cv", "irbo_mean", "topic_quality"]].to_string(index=False))
    print()


LDA TEMPORAL ANALYSIS: FINAL RESULTS

────────────────────────────────────────────────────────────
  Subject:        CS
  Num topics:     75
  Years:          26
  Coherence:      0.5056 +/- 0.0522
  IRBO:           0.9932 +/- 0.0056
  Topic Quality:  0.6685 +/- 0.0451
  Trends:         ↑30 growing, →23 stable, ↓21 declining
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Subject:        MATH
  Num topics:     50
  Years:          26
  Coherence:      0.4801 +/- 0.0510
  IRBO:           0.9816 +/- 0.0177
  Topic Quality:  0.6428 +/- 0.0435
  Trends:         ↑19 growing, →12 stable, ↓19 declining
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Subject:        PHYSICS
  Num topics:     50
  Years:          26
  Coherence:      0.4967 +/- 0.0745
  IRBO:           0.9942 +/- 0.0026
  Topic Quality:  0.6591 +/- 0.0666
  Trends:         ↑1